In [0]:
%sql
create schema if not exists apexlife.bronze

In [0]:
# Path to the source CSV file containing raw diagnosis data
source_path = 'abfss://data@apexlife.dfs.core.windows.net/staging/diagnosis/'

# Path for storing streaming checkpoint information
checkpoint_path = "abfss://data@apexlife.dfs.core.windows.net/bronze/diagnosis_raw/checkpoint/"

# Path for storing schema information inferred from the data
schema_location = "abfss://data@apexlife.dfs.core.windows.net/bronze/diagnosis_raw/schema/"

In [0]:
# Read streaming CSV files from cloud storage using Auto Loader
df = (
    spark.readStream.format('cloudFiles')                  # Use Auto Loader for efficient file discovery
    .option("cloudFiles.format", "csv")                    # Specify the file format as CSV
    .option("header", "true")                              # Indicate that CSV files contain a header row
    .option("inferSchema", "true")                         # Automatically infer the schema from the data
    .option("cloudFiles.schemaLocation", schema_location)  # Path to store the inferred schema
    .option("cloudFiles.maxFilesPerTrigger", 1)            # Process one file per trigger for controlled ingestion
    .load(source_path)                                     # Load files from the specified source path
)

In [0]:
# Drop the '_rescued_data' column from the streaming DataFrame before writing
(
    df.drop('_rescued_data')
        .writeStream
        .format("delta")                                    # Write the stream in Delta Lake format
        .option("checkpointLocation", checkpoint_path)      # Specify the checkpoint location for fault tolerance
        .outputMode("append")                               # Use 'append' mode to write only new rows
        .trigger(availableNow=True)                         # Process all available data and then stop
        .toTable("apexlife.bronze.diagnosis_raw")           # Write the stream output to the specified Delta table
)